In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# ======================== sort_contours ===========================================
def sort_contours(cnts, method="left-to-right"):
    reverse = False
    i = 0
    if method == "right-to-left" or method == "bottom-to-top":
        reverse = True
    if method == "top-to-bottom" or method == "bottom-to-top":
        i = 1
    boundingBoxes = [cv2.boundingRect(c) for c in cnts]
    (cnts, boundingBoxes) = zip(*sorted(zip(cnts, boundingBoxes),
            key=lambda b:b[1][i], reverse=reverse))
    return (cnts, boundingBoxes)
# ==================================================================================

def box_extraction(img_for_box_extraction_path, cropped_dir_path, 
                   min_line_length, max_line_gap, threshold):
    
    # 1. Читаем изображение в градациях серого
    img = cv2.imread(img_for_box_extraction_path, 0)
    
    # 2. Бинаризация (ОТСУ) и инверсия (чтобы линии стали белыми на черном фоне)
    (thresh, img_bin) = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    img_bin = 255 - img_bin
    cv2.imwrite("inter_processing/Image_bin.jpg", img_bin)

    # 3. HoughLinesP - ищем прямые линии (ГОРИЗОНТАЛЬНЫЕ И ВЕРТИКАЛЬНЫЕ)
    # Используем стандартный Canny для поиска границ перед Hough
    edges = cv2.Canny(img_bin, 50, 150, apertureSize=3)
    
    lines = cv2.HoughLinesP(
        edges,
        rho=1,                     # Разрешение по расстоянию (в пикселях)
        theta=np.pi / 180,         # Разрешение по углу (в радианах)
        threshold=threshold,       # Порог накопления голосов (чем выше, тем строже поиск)
        minLineLength=min_line_length, # Минимальная длина линии (игнорируем короткий мусор)
        maxLineGap=max_line_gap        # ГЛАВНОЕ: Максимальный разрыв, который нужно соединить!
    )

    # 4. Создаем пустое черное изображение, чтобы нарисовать найденные линии
    lines_mask = np.zeros_like(img_bin)
    
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Рисуем линию на маске (белым цветом)
            cv2.line(lines_mask, (x1, y1), (x2, y2), 255, 2)

    cv2.imwrite("inter_processing/lines_mask.jpg", lines_mask)

    # 5. Находим контуры на основе нарисованных линий
    contours, hierarchy = cv2.findContours(lines_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    (contours, boundingBoxes) = sort_contours(contours, method="top-to-bottom")
    
    # 6. Создаем изображение для визуализации (с зелеными рамками)
    vis_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    
    idx = 0
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        
        # Фильтр боксов (как у вас в коде)
        if (w > 80 and h > 20) and w > 3*h:
            idx += 1
            new_img = img[y:y+h, x:x+w]
            cv2.imwrite(cropped_dir_path + str(idx) + '.png', new_img)
            # Рисуем зеленую рамку
            cv2.rectangle(vis_img, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # Текст на картинке
    text = f"MinLen: {min_line_length}, Gap: {max_line_gap}, Thresh: {threshold} | Boxes: {idx}"
    cv2.putText(vis_img, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    return cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB)

# ======================= CALLBACK FUNCTION ============================
def update_image(change):
    with output_ui:
        clear_output(wait=True) # Очищаем предыдущий график
        
        min_len = slider_min_len.value
        max_gap = slider_max_gap.value
        thresh = slider_thresh.value
        
        # Вызываем функцию
        result_img = box_extraction(
            "/media/vadim/1TB_SSD/my_github/computer-vision-document-table-parser/input_images/1.jpg", 
            "./output/", 
            min_len, max_gap, thresh
        )
        
        # Показываем в Jupyter
        plt.figure(figsize=(15, 10))
        plt.imshow(result_img)
        plt.axis('off')
        plt.show()

# ======================= SETUP DIRECTORIES ============================
Path("./output/").mkdir(parents=True, exist_ok=True)
Path("./inter_processing/").mkdir(parents=True, exist_ok=True)

# ======================= CREATE WIDGETS ===============================
# ГЛАВНЫЕ ПАРАМЕТРЫ ДЛЯ HOUGHLINESP:
slider_min_len = widgets.IntSlider(value=80, min=20, max=300, step=5, description='Min Line Len:')
slider_max_gap = widgets.IntSlider(value=20, min=2, max=100, step=2, description='Max Line Gap:') # ЭТО СОЕДИНЯЕТ РАЗРЫВЫ
slider_thresh = widgets.IntSlider(value=50, min=10, max=200, step=5, description='Threshold:')

output_ui = widgets.Output()

# ==================== BIND EVENTS & SHOW ====================
slider_min_len.observe(update_image, names='value')
slider_max_gap.observe(update_image, names='value')
slider_thresh.observe(update_image, names='value')

display(widgets.VBox([slider_min_len, slider_max_gap, slider_thresh, output_ui]))

update_image(None)